# M4.5: multilingual reranker benchmark

This notebook compares Dense, Hybrid RRF, Dense top-20 + reranker, and Hybrid RRF top-20 + reranker. The cross-encoder's relevance score is the final order; original dense, BM25, and RRF scores are not mixed after reranking. If a CUDA failure occurred in this runtime, use **Runtime → Disconnect and delete runtime**, reopen the notebook, and Run All in a fresh session.

In [1]:
import os
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = 'https://github.com/ozgemelteminan/prompt-generator-rag'  # Replace this URL.
REPOSITORY_REF = 'main'  # Branch, tag, or commit to benchmark.
repository = Path('prompt-generator-rag')
if not repository.exists():
    subprocess.run(['git', 'clone', REPOSITORY_URL], check=True)
else:
    subprocess.run(['git', '-C', str(repository), 'fetch', '--all', '--tags', '--prune'], check=True)
subprocess.run(['git', '-C', str(repository), 'checkout', REPOSITORY_REF], check=True)
branch = subprocess.run(['git', '-C', str(repository), 'branch', '--show-current'], check=True, capture_output=True, text=True).stdout.strip()
if branch:
    subprocess.run(['git', '-C', str(repository), 'pull', '--ff-only', 'origin', branch], check=True)
os.chdir(repository)
subprocess.run(['pip', 'install', '-q', '--upgrade', 'transformers==4.57.6', 'sentence-transformers==5.6.0'], check=True)
subprocess.run(['pip', 'install', '-q', '-e', 'packages/prompt-engine'], check=True)
subprocess.run(['pip', 'install', '-q', '-e', 'apps/api', '--no-deps'], check=True)

import torch
import transformers
import sentence_transformers
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print('GPU:', GPU_NAME)
print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('sentence-transformers:', sentence_transformers.__version__)
assert transformers.__version__ == '4.57.6'
RUNTIME_METADATA = {'torchVersion': torch.__version__, 'transformersVersion': transformers.__version__, 'sentenceTransformersVersion': sentence_transformers.__version__, 'cudaDevice': GPU_NAME}

repository_root = Path.cwd().resolve()
api_root = repository_root / 'apps' / 'api'
for import_root in (repository_root, api_root):
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))
stale_modules = [name for name in sys.modules if name == 'app' or name.startswith('app.') or name == 'evals' or name.startswith('evals.')]
if stale_modules:
    raise RuntimeError('Stale modules are loaded. Restart the runtime, then Run All.')

GPU: Tesla T4
torch: 2.11.0+cu128
transformers: 4.57.6
sentence-transformers: 5.6.0


In [2]:
import pandas as pd
from evals.src.dataset import load_dataset
from evals.src.embedding_eval import SentenceTransformerEmbeddingAdapter, embedding_model_registry, frozen_production_chunks
from evals.src.reranker_eval import CANDIDATE_DEPTH, CrossEncoderReranker, RERANKER_MODEL_ID, run_reranker_benchmark, save_reranker_results

ROOT = Path.cwd()
dataset = load_dataset(ROOT / 'evals/datasets/retrieval_eval_v1.json')
chunks = frozen_production_chunks(dataset)  # Generated once with 350/500/40 and reused by every system.
e5_spec = embedding_model_registry()['multilingual_e5_large_instruct']
assert e5_spec.model_id == 'intfloat/multilingual-e5-large-instruct'
dense_adapter = SentenceTransformerEmbeddingAdapter(e5_spec)
reranker = CrossEncoderReranker(RERANKER_MODEL_ID)
try:
    evaluation = run_reranker_benchmark(dataset, chunks=chunks, adapter=dense_adapter, reranker=reranker, candidate_depth=CANDIDATE_DEPTH)
finally:
    dense_adapter.release()
    reranker.release()
save_reranker_results(evaluation, dataset_version=dataset.version, output_dir=ROOT / 'evals/results/reranking', runtime_metadata=RUNTIME_METADATA)
results = evaluation.results

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_xlm-roberta_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

In [3]:
rows = []
for result in results:
    row = {'Retriever': result.retriever, **result.metrics}
    row['TR MRR'] = result.by_language.get('tr', {}).get('mrr', 0.0)
    row['EN MRR'] = result.by_language.get('en', {}).get('mrr', 0.0)
    rows.append(row)
comparison = pd.DataFrame(rows)
display(comparison)

by_key = {result.retriever_key: result for result in results}
deltas = []
for candidate, baseline, label in [('dense_reranker', 'dense_e5', 'Dense+Reranker - Dense'), ('hybrid_reranker', 'hybrid_rrf', 'Hybrid+Reranker - Hybrid'), ('hybrid_reranker', 'dense_reranker', 'Hybrid+Reranker - Dense+Reranker')]:
    deltas.append({'Comparison': label, **{metric: by_key[candidate].metrics[metric] - by_key[baseline].metrics[metric] for metric in by_key[candidate].metrics}})
display(pd.DataFrame(deltas))

category_rows = []
for category in ['hard_paraphrase', 'terminology_mismatch', 'morphology_heavy', 'near_negative', 'same_topic_competitor', 'multi_section', 'cross_paragraph']:
    category_rows.append({'Category': category, **{result.retriever: result.by_category.get(category, {}).get('mrr', 0.0) for result in results}})
display(pd.DataFrame(category_rows))

for metric, label in [('recall_at_5', 'Recall@5'), ('recall_at_10', 'Recall@10'), ('mrr', 'MRR'), ('ndcg_at_10', 'nDCG@10'), ('hit_rate_at_5', 'HitRate@5'), ('required_block_coverage_at_5', 'BlockCoverage@5'), ('required_block_coverage_at_10', 'BlockCoverage@10')]:
    winners = comparison.loc[comparison[metric] == comparison[metric].max(), 'Retriever'].tolist()
    print(f'Best {label}: {winners}')

for signal, query_ids in evaluation.diagnostics.items():
    print(f'{signal}: {len(query_ids)} queries', query_ids)

,Retriever,recall_at_5,recall_at_10,hit_rate_at_5,mrr,ndcg_at_10,required_block_coverage_at_5,required_block_coverage_at_10,TR MRR,EN MRR
0,Dense — intfloat/multilingual-e5-large-instruct,0.958333,1.000000,0.964286,0.870040,0.851617,0.958333,1.000000,0.940476,0.799603
1,Hybrid — Dense + BM25 RRF,0.958333,1.000000,0.964286,0.877395,0.856806,0.958333,1.000000,0.961905,0.792885
2,Dense + Reranker,0.898810,0.982143,0.904762,0.873720,0.844827,0.898810,0.982143,0.972222,0.775217
3,Hybrid RRF + Reranker,0.898810,0.982143,0.904762,0.873720,0.844827,0.898810,0.982143,0.972222,0.775217


,Comparison,recall_at_5,recall_at_10,hit_rate_at_5,mrr,ndcg_at_10,required_block_coverage_at_5,required_block_coverage_at_10
0,Dense+Reranker - Dense,-0.059524,-0.017857,-0.059524,0.003680,-0.006790,-0.059524,-0.017857
1,Hybrid+Reranker - Hybrid,-0.059524,-0.017857,-0.059524,-0.003675,-0.011979,-0.059524,-0.017857
2,Hybrid+Reranker - Dense+Reranker,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


,Category,Dense — intfloat/multilingual-e5-large-instruct,Hybrid — Dense + BM25 RRF,Dense + Reranker,Hybrid RRF + Reranker
0,hard_paraphrase,1.000000,0.958333,0.958333,0.958333
1,terminology_mismatch,0.513889,0.511111,0.464286,0.464286
2,morphology_heavy,1.000000,1.000000,1.000000,1.000000
3,near_negative,0.805556,0.833333,0.770833,0.770833
4,same_topic_competitor,0.805556,0.777778,0.718519,0.718519
5,multi_section,0.916667,1.000000,1.000000,1.000000
6,cross_paragraph,0.652778,0.595833,0.625661,0.625661


Best Recall@5: ['Dense — intfloat/multilingual-e5-large-instruct', 'Hybrid — Dense + BM25 RRF']
Best Recall@10: ['Dense — intfloat/multilingual-e5-large-instruct', 'Hybrid — Dense + BM25 RRF']
Best MRR: ['Hybrid — Dense + BM25 RRF']
Best nDCG@10: ['Hybrid — Dense + BM25 RRF']
Best HitRate@5: ['Dense — intfloat/multilingual-e5-large-instruct', 'Hybrid — Dense + BM25 RRF']
Best BlockCoverage@5: ['Dense — intfloat/multilingual-e5-large-instruct', 'Hybrid — Dense + BM25 RRF']
Best BlockCoverage@10: ['Dense — intfloat/multilingual-e5-large-instruct', 'Hybrid — Dense + BM25 RRF']
dense_reranker_improves: 6 queries ['m42-security-02', 'm42-security-04', 'm42-security-14', 'm42-retrieval-10', 'm42-retrieval-11', 'm42-code-02']
dense_reranker_hurts: 10 queries ['m42-retrieval-03', 'm42-retrieval-04', 'm42-retrieval-09', 'm42-code-04', 'm42-code-07', 'm42-code-08', 'm42-planning-04', 'm42-planning-07', 'm42-planning-09', 'm42-planning-14']
dense_rescued_into_top_5: 1 queries ['m42-security-04']


In [4]:
from google.colab import files

files.download(
    "/content/prompt-generator-rag/evals/results/reranking/reranker_results_v1.json"
)
files.download(
    "/content/prompt-generator-rag/evals/results/reranking/reranker_results_v1.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>